# 1. importacion y inspeccion inicial 

In [3]:
import pandas as pd

df = pd.read_csv("../../data/novamarket_features_seleccionadas.csv")
df.head()

,nivel_lealtad_cod,precio_unitario,edad_cliente,monto_compra_log,canal_App,dias_desde_ultima_compra,total_gastado,fuga_cliente,id_cliente
0,3.0,475600.0,47.0,13.072335,False,340.0,832600.0,0,C33878
1,3.0,341900.0,27.0,12.742277,False,182.0,4296800.0,0,C87381
2,2.0,280900.0,45.0,12.545758,False,179.0,3045100.0,0,C34664
3,1.0,523400.0,63.0,13.168103,False,316.0,1721600.0,1,C106247
4,2.0,261600.0,51.0,13.167721,False,190.0,1488300.0,0,C60292


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 570976 entries, 0 to 570975
Data columns (total 9 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   nivel_lealtad_cod         570976 non-null  float64
 1   precio_unitario           570976 non-null  float64
 2   edad_cliente              570976 non-null  float64
 3   monto_compra_log          570976 non-null  float64
 4   canal_App                 570976 non-null  bool   
 5   dias_desde_ultima_compra  569878 non-null  float64
 6   total_gastado             570976 non-null  float64
 7   fuga_cliente              570976 non-null  int64  
 8   id_cliente                570976 non-null  str    
dtypes: bool(1), float64(6), int64(1), str(1)
memory usage: 35.4 MB


In [5]:
print(df["nivel_lealtad_cod"].value_counts().sort_index())
print(df["canal_App"].value_counts())
print(df["dias_desde_ultima_compra"].isna().sum())

nivel_lealtad_cod
0.0    143220
1.0    142526
2.0    142916
3.0    142314
Name: count, dtype: int64
canal_App
False    428640
True     142336
Name: count, dtype: int64
1098


## Paso 2 uu

### Paso 2.1 — Revisar los valores únicos de cada columna

**Qué hace:** recorre todas las columnas y muestra los primeros 5 valores únicos de cada una.

**Para qué la necesitamos:** el `ColumnTransformer` aplica un tratamiento distinto según el tipo de columna (imputar y escalar las continuas, dejar pasar las binarias), así que antes de armarlo hay que ver con qué tipo de datos trabajamos. Un vistazo a los valores únicos distingue una binaria (solo `0`/`1` o `False`/`True`) de una continua (muchos valores distintos con decimales).

In [6]:
for col in df.columns:
    print(col, df[col].unique()[:5])

nivel_lealtad_cod [3. 2. 1. 0.]
precio_unitario [475600. 341900. 280900. 523400. 261600.]
edad_cliente [47. 27. 45. 63. 51.]
monto_compra_log [13.07233455 12.7422765  12.54575757 13.16810318 13.16772099]
canal_App [False  True]
dias_desde_ultima_compra [340. 182. 179. 316. 190.]
total_gastado [ 832600. 4296800. 3045100. 1721600. 1488300.]
fuga_cliente [0 1]
id_cliente <StringArray>
['C33878', 'C87381', 'C34664', 'C106247', 'C60292']
Length: 5, dtype: str


### Paso 2.2 — Separar las columnas en dos listas según su naturaleza

**Qué hace:** crea `columnas_continuas` (numéricas con decimales o rangos amplios, más la ordinal `nivel_lealtad_cod` de la semana 9) y `columnas_binarias` (las que solo valen 0 y 1, salidas del one-hot de la semana 9). `id_cliente` y `fuga_cliente` quedan fuera: la primera es un identificador y la segunda es la variable objetivo (`y`).

**Para qué la necesitamos:** cada lista se convertirá en una rama del `ColumnTransformer`. Las continuas pasarán por imputación y escalado; las binarias pasarán sin cambios (`"passthrough"`), porque escalarlas convertiría un `1` legible en un número como `1.34` sin ganar información.

In [7]:
columnas_continuas = [
    "nivel_lealtad_cod",          # ordinal (0-3) de la semana 9
    "precio_unitario",
    "edad_cliente",
    "monto_compra_log",
    "dias_desde_ultima_compra",   # única con nulos (1.098)
    "total_gastado",
]

columnas_binarias = [
    "canal_App",                  # one-hot de la semana 9
]

### Paso 2.3 — Verificar que las dos listas cubren todas las features

**Qué hace:** compara `len(columnas_continuas) + len(columnas_binarias)` con el número de columnas de `df` menos `fuga_cliente` e `id_cliente`, y además comprueba que ninguna columna aparece en ambas listas.

**Para qué la necesitamos:** si olvidamos una columna, el `ColumnTransformer` la descartaría en silencio (por defecto `remainder="drop"`) y el modelo se entrenaría sin una feature de la selección de S14. Esta verificación convierte ese error invisible en un `assert` que falla de inmediato.

In [8]:
n_esperado = df.shape[1] - 2   # menos fuga_cliente e id_cliente
n_listas = len(columnas_continuas) + len(columnas_binarias)

print(n_listas, n_esperado)
assert n_listas == n_esperado, "Falta o sobra alguna columna"
assert not set(columnas_continuas) & set(columnas_binarias), "Hay columnas repetidas"

7 7


## Paso 3 

### Paso 3.1 — Sub-pipeline para las columnas continuas

**Qué hace:** importa `Pipeline`, `SimpleImputer` y `StandardScaler`, y arma `pipeline_continuas` con dos pasos en orden: primero el imputador (rellena nulos con la mediana de cada columna) y luego el escalador (centra en 0 y deja desviación 1).

**Para qué la necesitamos:** es la receta que se aplicará solo a `columnas_continuas`. El orden importa: se imputa antes de escalar para que el escalador nunca vea un `NaN` y para que la media y la desviación se calculen con valores completos. Al estar dentro de un `Pipeline`, ambos pasos aprenden sus parámetros (medianas, media, desviación) solo con los datos de entrenamiento, lo que evita la fuga de datos.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipeline_continuas = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
])

### Paso 3.2 — Verificar que el sub-pipeline quedó bien armado

**Qué hace:** muestra los pasos del pipeline por nombre.

**Para qué la necesitamos:** confirma que existen las claves `"imputador"` y `"escalador"`. Esos nombres se usarán más adelante para inspeccionar los parámetros aprendidos (por ejemplo, las medianas de cada columna) y para documentar la versión en `S16_documento_version_pipeline.md`.

In [10]:
pipeline_continuas.named_steps

{'imputador': SimpleImputer(strategy='median'), 'escalador': StandardScaler()}

## Paso 4

### Paso 4.1 — Unir las dos ramas en un `ColumnTransformer`

**Qué hace:** crea `preprocesador`, que aplica un tratamiento distinto a cada grupo de columnas: `pipeline_continuas` (imputar con mediana y escalar) a `columnas_continuas`, y `"passthrough"` (dejar pasar sin cambios) a `columnas_binarias`.

**Para qué la necesitamos:** un `Pipeline` simple aplicaría los mismos pasos a todas las columnas, y escalar `canal_App` no tiene sentido. El `ColumnTransformer` resuelve eso: cada columna recibe solo el tratamiento que le corresponde. Las columnas que no están en ninguna lista (`id_cliente`, `fuga_cliente`) se descartan por defecto (`remainder="drop"`), que es lo que queremos porque no son features.

In [11]:
from sklearn.compose import ColumnTransformer

preprocesador = ColumnTransformer(transformers=[
    ("continuas", pipeline_continuas, columnas_continuas),
    ("binarias", "passthrough", columnas_binarias),
])

### Paso 4.2 — Verificar que el preprocesador referencia columnas existentes

**Qué hace:** muestra los transformadores registrados y comprueba que todos los nombres de columna que usa existen en `df`.

**Para qué la necesitamos:** crear el `ColumnTransformer` no valida los nombres; el error de "columna no encontrada" solo aparece al ajustarlo (`fit`). Esta comprobación adelanta ese fallo y deja claro qué columnas descarta el preprocesador.

In [12]:
print(preprocesador.transformers)

usadas = columnas_continuas + columnas_binarias
faltantes = [c for c in usadas if c not in df.columns]
descartadas = [c for c in df.columns if c not in usadas]

print("Faltantes en df:", faltantes)
print("Descartadas por el preprocesador:", descartadas)
assert not faltantes

[('continuas', Pipeline(steps=[('imputador', SimpleImputer(strategy='median')),
                ('escalador', StandardScaler())]), ['nivel_lealtad_cod', 'precio_unitario', 'edad_cliente', 'monto_compra_log', 'dias_desde_ultima_compra', 'total_gastado']), ('binarias', 'passthrough', ['canal_App'])]
Faltantes en df: []
Descartadas por el preprocesador: ['fuga_cliente', 'id_cliente']


## Paso 5

### Paso 5.1 — Separar las features y probar el preprocesador con 20 filas

**Qué hace:** crea `X` (todas las columnas menos `fuga_cliente` e `id_cliente`) y ajusta y aplica el `preprocesador` solo a las primeras 20 filas.

**Para qué la necesitamos:** es una prueba de humo. Con 20 filas el error aparece en segundos, no después de procesar 570.976 filas. Nos dice si el `ColumnTransformer` está bien armado antes de usarlo con validación cruzada. Esta salida es solo de prueba y no se usa para el modelo, porque con 20 filas las medianas y escalas no son representativas.

In [13]:
X = df.drop(columns=["fuga_cliente", "id_cliente"])

X_transformado_prueba = preprocesador.fit_transform(X.head(20))
print(X_transformado_prueba.shape)

(20, 7)


### Paso 5.2 — Verificar que todas las columnas llegaron a la salida

**Qué hace:** compara el número de columnas de salida con la suma de las dos listas, y revisa el tipo de dato y que no queden nulos.

**Para qué la necesitamos:** si falta una columna, alguna feature se perdió en silencio. Revisar el `dtype` descarta que `canal_App` (tipo `bool`) haya dejado la salida como `object`, y revisar los nulos confirma que el imputador cumplió su trabajo.

In [14]:
esperado = len(columnas_continuas) + len(columnas_binarias)
print(X_transformado_prueba.shape[1], esperado)
assert X_transformado_prueba.shape[1] == esperado

print("dtype:", X_transformado_prueba.dtype)
print("nulos:", pd.isna(X_transformado_prueba).sum())
print(X_transformado_prueba[:3].round(2))

7 7
dtype: float64
nulos: 0
[[ 1.55  0.49  0.12  0.28  0.77 -1.39  0.  ]
 [ 1.55 -0.41 -1.05  0.2  -0.93  1.05  0.  ]
 [ 0.61 -0.82  0.    0.15 -0.96  0.17  0.  ]]


## Paso 6 

### Paso 6.1 — Envolver el `ColumnTransformer` dentro de un `Pipeline` general

**Qué hace:** crea `pipeline_novamarket` con dos pasos en orden: primero el `preprocesador` (imputar y escalar las continuas, dejar pasar las binarias) y luego una `LogisticRegression` simple.

**Para qué la necesitamos:** hasta ahora el preprocesador solo transforma datos, y no hay forma de saber si lo hace sin fuga. Al ponerle un modelo al final, `cross_val_score` puede ajustar todo el pipeline dentro de cada fold: las medianas, medias y desviaciones se calculan solo con el entrenamiento de esa ronda. La regresión logística **no es el modelo final** del equipo de Machine Learning: es un modelo de validación, un termómetro para medir el preprocesamiento. `class_weight="balanced"` pondera más a los clientes que se fugan (cerca del 21 %), y como se calcula solo con el entrenamiento de cada ronda, no produce fuga.

In [15]:
from sklearn.linear_model import LogisticRegression

pipeline_novamarket = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("modelo_validacion", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

### Paso 6.2 — Verificar que el pipeline general quedó bien armado

**Qué hace:** muestra los pasos del pipeline por nombre.

**Para qué la necesitamos:** confirma que existen las claves `"preprocesador"` y `"modelo_validacion"`. Más adelante, `pipeline_novamarket[:-1]` devolverá solo el preprocesamiento (para procesar el lote de octubre) y `pipeline_novamarket.named_steps` servirá para documentar la versión en `S16_documento_version_pipeline.md`.

In [16]:
pipeline_novamarket.named_steps

{'preprocesador': ColumnTransformer(transformers=[('continuas',
                                  Pipeline(steps=[('imputador',
                                                   SimpleImputer(strategy='median')),
                                                  ('escalador',
                                                   StandardScaler())]),
                                  ['nivel_lealtad_cod', 'precio_unitario',
                                   'edad_cliente', 'monto_compra_log',
                                   'dias_desde_ultima_compra',
                                   'total_gastado']),
                                 ('binarias', 'passthrough', ['canal_App'])]),
 'modelo_validacion': LogisticRegression(class_weight='balanced', max_iter=1000)}

## Paso 7

### Paso 7.1 — Separar las features (`X`) de la variable objetivo (`y`)

**Qué hace:** `X` contiene las 7 features; `y` contiene `fuga_cliente`, lo que el modelo debe predecir. `id_cliente` queda fuera de `X` porque es un identificador, no una característica del cliente.

**Para qué la necesitamos:** `train_test_split` y `cross_val_score` necesitan las features y el objetivo por separado. Si `fuga_cliente` se quedara en `X`, el modelo vería la respuesta (fuga de datos).

In [17]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["fuga_cliente", "id_cliente"])
y = df["fuga_cliente"]

### Paso 7.2 — Dividir en entrenamiento (80 %) y prueba (20 %)

**Qué hace:** reserva el 20 % de las filas como conjunto de prueba. `random_state=42` hace la división reproducible, y `stratify=y` mantiene la misma proporción de fuga (≈21 %) en ambos conjuntos.

**Para qué la necesitamos:** el conjunto de prueba simula los datos nuevos (como el lote de octubre): el pipeline nunca debe ajustarse con él. Sin `stratify`, la proporción de clientes que se fugan podría variar entre conjuntos y distorsionar las métricas.

In [18]:
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_entrenamiento.value_counts(normalize=True))
print(y_prueba.value_counts(normalize=True))
print(X_entrenamiento.shape, X_prueba.shape)

fuga_cliente
0    0.790891
1    0.209109
Name: proportion, dtype: float64
fuga_cliente
0    0.790886
1    0.209114
Name: proportion, dtype: float64
(456780, 7) (114196, 7)


### Paso 7.3 — Revisar si un mismo cliente aparece en entrenamiento y en prueba

**Qué hace:** cuenta cuántos `id_cliente` aparecen en ambos conjuntos.

**Para qué la necesitamos:** el dataset tiene una fila por pedido, no por cliente (570.976 filas frente a ≈151.000 clientes únicos). Features como `total_gastado` o `dias_desde_ultima_compra` valen lo mismo para todas las filas de un cliente. Si un cliente cae en ambos conjuntos, el modelo "reconoce" en la prueba a clientes que ya vio en el entrenamiento, y las métricas salen optimistas. Esto no lo detecta el pipeline, hay que medirlo.

In [19]:
ids_ent = set(df.loc[X_entrenamiento.index, "id_cliente"])
ids_pru = set(df.loc[X_prueba.index, "id_cliente"])
compartidos = ids_ent & ids_pru

print("Clientes únicos en entrenamiento:", len(ids_ent))
print("Clientes únicos en prueba:", len(ids_pru))
print("Clientes en ambos:", len(compartidos))

Clientes únicos en entrenamiento: 146889
Clientes únicos en prueba: 80747
Clientes en ambos: 76519


Es esperable, no un error. La división por filas reparte los pedidos de un mismo cliente entre ambos conjuntos, y casi todos los clientes tienen varios pedidos.

## Paso 8

### Paso 8.0 — Validación cruzada del pipeline completo

**Qué hace:** divide el conjunto de entrenamiento en 5 partes. En cada ronda ajusta todo el pipeline (imputación, escalado y modelo) con 4 partes y mide el `recall` en la parte restante. Devuelve 5 valores, uno por ronda.

**Para qué la necesitamos:** como el preprocesamiento está dentro del `Pipeline`, las medianas, medias y desviaciones se recalculan en cada ronda solo con su entrenamiento. Así se mide el pipeline sin fuga de datos en el preprocesamiento. Usamos `recall` porque nuestro objetivo es detectar a los clientes que se fugan.

In [20]:
from sklearn.model_selection import cross_val_score

resultados = cross_val_score(
    pipeline_novamarket, X_entrenamiento, y_entrenamiento,
    cv=5, scoring="recall"
)
print(resultados)

[0.65057844 0.64984557 0.64246453 0.65237647 0.64389657]


### Paso 8.1 — Promedio y desviación estándar del `recall`

**Qué hace:** resume los 5 valores de la validación cruzada en dos números: el promedio (qué tan bien detecta fugas en general) y la desviación estándar (qué tan estable es entre rondas).

**Para qué la necesitamos:** un solo valor de `recall` podría deberse a una partición afortunada. El promedio con su desviación dice cuánto confiar en el resultado.

### Paso 8.2 — Promedio y desviación estándar del `recall`

**Qué hace:** resume los 5 valores de la validación cruzada en dos números: el promedio (qué tan bien detecta fugas en general) y la desviación estándar (qué tan estable es entre rondas).

**Para qué la necesitamos:** un solo valor de `recall` podría deberse a una partición afortunada. El promedio con su desviación dice cuánto confiar en el resultado.

In [21]:
print("Recall promedio:", resultados.mean().round(4))
print("Desviación estándar:", resultados.std().round(4))

Recall promedio: 0.6478
Desviación estándar: 0.0039


### Resultado de la validación cruzada

El pipeline obtuvo un `recall` promedio de **0,648** con una desviación estándar de **0,004** en 5 particiones, usando una regresión logística con `class_weight="balanced"` como modelo de validación. Esto significa que, de cada 100 clientes que realmente se fugan, el modelo detecta unos 65, y que el resultado es muy estable: el `recall` varía menos de un punto porcentual entre particiones (0,642 a 0,652). Como el preprocesamiento (imputación y escalado) está dentro del pipeline y se reajusta en cada partición solo con su entrenamiento, el pipeline no tiene fuga de datos en esa etapa. Para el objetivo de detectar clientes en fuga el resultado es razonable para un modelo simple, ya que atrapa a cerca de dos de cada tres, aunque deja sin detectar a aproximadamente un tercio. Este valor puede estar algo optimista, porque la partición se hizo por filas y los mismos clientes aparecen en entrenamiento y validación.

## Paso 9 

### Paso 9.1 — Ajustar el pipeline con el conjunto de entrenamiento

**Qué hace:** entrena el pipeline completo con `X_entrenamiento` y `y_entrenamiento`. El imputador aprende la mediana de cada columna continua, el escalador aprende su media y desviación, y la regresión logística aprende sus coeficientes. Todo se calcula únicamente con las filas de entrenamiento.

**Para qué la necesitamos:** hasta ahora el pipeline era solo una receta (y `cross_val_score` ajustaba copias temporales). Este es el objeto definitivo, con los parámetros aprendidos adentro. Es el que se guardará con `joblib` para procesar el lote de octubre. El conjunto de prueba no interviene: sigue sin usarse, como datos que el pipeline nunca ha visto.

In [22]:
pipeline_novamarket.fit(X_entrenamiento, y_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesador', ...), ('modelo_validacion', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['nivel_lealtad_cod','precio_unitario','edad_cliente',...,'canal_App', 'dias_desde_ultima_compra','total_gastado']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('continuas', ...), ('binarias', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defau

### Paso 9.2 — Verificar que el preprocesador quedó ajustado

**Qué hace:** muestra el `ColumnTransformer` ya ajustado y las medianas, medias y desviaciones que aprendió de los datos de entrenamiento.

**Para qué la necesitamos:** confirma que el pipeline guardó sus parámetros (el `fit` realmente ocurrió) y deja a la vista los valores que se aplicarán a todo lote nuevo. Esos números pertenecen a esta versión del pipeline y se citarán en `S16_documento_version_pipeline.md`.

In [23]:
pipeline_novamarket.named_steps["preprocesador"]

cont = pipeline_novamarket.named_steps["preprocesador"].named_transformers_["continuas"]
resumen = pd.DataFrame({
    "mediana_imputacion": cont.named_steps["imputador"].statistics_,
    "media_escalador": cont.named_steps["escalador"].mean_,
    "desviacion_escalador": cont.named_steps["escalador"].scale_,
}, index=columnas_continuas)
print(resumen.round(4))

                          mediana_imputacion  media_escalador  \
nivel_lealtad_cod               1.000000e+00     1.498300e+00   
precio_unitario                 3.125000e+05     3.445636e+05   
edad_cliente                    4.700000e+01     4.651970e+01   
monto_compra_log                1.303220e+01     1.215110e+01   
dias_desde_ultima_compra        2.340000e+02     2.624409e+02   
total_gastado                   2.776800e+06     3.075738e+06   

                          desviacion_escalador  
nivel_lealtad_cod                 1.117900e+00  
precio_unitario                   1.964827e+05  
edad_cliente                      1.624290e+01  
monto_compra_log                  3.506800e+00  
dias_desde_ultima_compra          9.551690e+01  
total_gastado                     1.860830e+06  


## 10 Confirmación del ajuste (`fit`)

El pipeline se ajustó **una sola vez**, en el paso donde se ejecutó `pipeline_novamarket.fit(X_entrenamiento, y_entrenamiento)` (Paso 10.1), y **únicamente** con `X_entrenamiento` y `y_entrenamiento`. Las medianas, medias, desviaciones y coeficientes que guarda el pipeline salen todos de esas filas. El conjunto de prueba (`X_prueba`, `y_prueba`) no intervino en ningún ajuste.

### Evaluación en el conjunto de prueba

**Qué hace:** usa el pipeline ya ajustado para predecir sobre `X_prueba` y calcula el `recall` contra `y_prueba`. Solo se usa `.predict()`: no se vuelve a ajustar nada.

**Para qué la necesitamos:** es una segunda medición, independiente de la validación cruzada, sobre filas que el pipeline no usó para aprender. Si el `recall` queda cerca del promedio de la validación cruzada (0,648), el resultado es consistente. Si saliera muy por encima, habría que sospechar de fuga. Se usa `recall_score` y no `.score()`, porque `.score()` devuelve la exactitud (`accuracy`), que no mide lo que nos interesa.

In [24]:
from sklearn.metrics import recall_score

recall_prueba = recall_score(y_prueba, pipeline_novamarket.predict(X_prueba))
print("Recall en prueba:", round(recall_prueba, 4))
print("Recall promedio en validación cruzada:", resultados.mean().round(4))

Recall en prueba: 0.6511
Recall promedio en validación cruzada: 0.6478


### Evaluar con la prueba vs. ajustar con la prueba

**Evaluar con la prueba** es dejar que el pipeline ya ajustado prediga sobre datos que no usó para aprender, para medir qué tan bien funcionaría con datos nuevos (correcto). **Ajustar con la prueba** es dejar que esos datos influyan en medianas, medias, escalas o coeficientes: la prueba deja de ser "datos nuevos" y las métricas salen optimistas (fuga de datos, lo que nunca debemos hacer).

## Paso 11

### Paso 11.1 — Importar `joblib`

**Qué hace:** carga la librería que sabe guardar objetos de Python (como un pipeline ajustado) en un archivo y volver a leerlos.

**Para qué la necesitamos:** el pipeline vive en la memoria del notebook y desaparece al cerrarlo. `joblib` lo convierte en un archivo, que es lo que permitirá procesar el lote de octubre sin volver a ejecutar nada de este notebook.

In [25]:
import joblib

### Paso 11.2 — Guardar el pipeline entrenado como versión 1

**Qué hace:** escribe `pipeline_novamarket` (con todos sus parámetros ya aprendidos: medianas, medias, desviaciones y coeficientes) en el archivo `novamarket_pipeline_v1.joblib`.

**Para qué la necesitamos:** es el producto de la semana. El número de versión va en el nombre porque un archivo como `pipeline_final2` no dice qué contiene. Si más adelante cambia algo (columnas, estrategia de imputación, nuevo ajuste), se guarda como `novamarket_pipeline_v2.joblib`, **sin sobrescribir** la v1.

In [26]:
joblib.dump(pipeline_novamarket, "novamarket_pipeline_v1.joblib")

['novamarket_pipeline_v1.joblib']

### Paso 11.3 — Verificar que el archivo guardado reproduce el mismo resultado

**Qué hace:** carga el archivo en un objeto nuevo y comprueba que sus predicciones sobre el conjunto de prueba son idénticas a las del pipeline original. También registra las versiones de las librerías con las que se creó.

**Para qué la necesitamos:** un archivo guardado que no se haya probado es una suposición. Esta celda confirma que el "botón" funciona de verdad. Las versiones de `scikit-learn` y `pandas` se anotarán en el documento de versión, porque un `.joblib` puede fallar o comportarse distinto si se abre con otra versión de scikit-learn.

In [27]:
import sklearn, numpy as np, os

pipeline_cargado = joblib.load("novamarket_pipeline_v1.joblib")

iguales = np.array_equal(
    pipeline_novamarket.predict(X_prueba),
    pipeline_cargado.predict(X_prueba),
)
print("Predicciones idénticas:", iguales)
print("Tamaño del archivo (MB):", round(os.path.getsize("novamarket_pipeline_v1.joblib") / 1e6, 3))
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__)

Predicciones idénticas: True
Tamaño del archivo (MB): 0.004
scikit-learn: 1.9.0 | pandas: 3.0.5


## Paso 12

### Paso 12.1 — Cargar el pipeline desde el archivo

**Qué hace:** lee `novamarket_pipeline_v1.joblib` y crea un objeto nuevo, `pipeline_cargado`, independiente de `pipeline_novamarket`.

**Para qué la necesitamos:** es exactamente lo que haría Camila (o cualquier analista) con el lote de octubre: abrir el archivo en un notebook nuevo, sin haber visto nunca los doce notebooks del semestre. Si `pipeline_cargado` funciona, el archivo contiene todo lo necesario.

In [28]:
pipeline_cargado = joblib.load("novamarket_pipeline_v1.joblib")

### Paso 12.2 — Comparar el `recall` del pipeline cargado con el del original

**Qué hace:** calcula el `recall` en la prueba con el pipeline cargado desde el archivo y lo compara con el que obtuvimos antes, con el pipeline que estaba en memoria.

**Para qué la necesitamos:** si ambos son exactamente iguales, el archivo conserva todo lo que el pipeline aprendió (medianas, medias, desviaciones y coeficientes). Cualquier diferencia indicaría que se guardó otro objeto o que el pipeline se volvió a ajustar después de guardarlo.

In [29]:
from sklearn.metrics import recall_score

recall_cargado = recall_score(y_prueba, pipeline_cargado.predict(X_prueba))

print("Recall con el pipeline original:", round(recall_prueba, 6))
print("Recall con el pipeline cargado: ", round(recall_cargado, 6))
assert recall_cargado == recall_prueba, "El archivo no reproduce el resultado original"

Recall con el pipeline original: 0.651131
Recall con el pipeline cargado:  0.651131
